# The Continuous Thought Machine (JAX Port) -- Tutorial 01: MNIST

Modern deep learning models ignore time as a core computational element. In contrast, the **Continuous Thought Machine (CTM)** introduces internal recurrence and neural synchronization to model *thinking as a temporal process*.

### Key Ideas

- **Internal Ticks**: The CTM runs over a self-generated temporal axis (independent of input), which we view as a dimension over which thought can unfold.
- **Neuron-Level Models**: Each neuron has a private MLP that processes its own history of pre-activations over time.
- **Synchronization as Representation**: CTMs compute neuron-to-neuron synchronization over time and use these signals for attention and output.

### Why It Matters

- Enables **interpretable, dynamic reasoning**
- Supports **adaptive compute** (e.g. more ticks for harder tasks)
- Works across tasks: classification, reasoning, memory, RL -- *without changing the core mechanisms*.

### About this Port

This notebook is a faithful port of the original PyTorch CTM tutorial to **JAX + Flax**. Key differences from the PyTorch version:

- All model parameters are explicit (Flax modules use explicit `setup()` with `self.param(...)` declarations).
- The recurrent loop uses a Python for-loop that is **unrolled at JIT trace time**, producing a single fused XLA computation. This avoids Python-level overhead at runtime while keeping the code readable.
- `jax.jit` is applied to the training and evaluation steps for hardware-accelerated execution.
- Multi-head attention is implemented manually with einsum and explicit Q/K/V/output projections for full JIT transparency.
- Data loading still uses `torchvision` for convenience, but tensors are converted to JAX arrays before entering the model.

----

### MNIST Classification

In this tutorial we walk through a simple example; training a CTM to classify MNIST digits. We cover:
- Defining the model
- Constructing the loss function
- Training
- Building visualization

### Environment Setup (Google Colab)

Run the cell below to install dependencies and verify GPU availability. On Colab with a GPU runtime, JAX will automatically detect and use the NVIDIA GPU. If running locally with a GPU, ensure you have the correct `jax[cuda]` wheel installed.

In [ ]:
# -- Colab: install dependencies --
# Uncomment the following lines when running on Google Colab.
# !pip install -q jax[cuda12] flax optax torchvision imageio seaborn tqdm scipy

import jax
print(f"JAX version : {jax.__version__}")
print(f"Devices     : {jax.devices()}")
print(f"Default backend: {jax.default_backend()}")

# Verify GPU is available
assert jax.default_backend() == "gpu", (
    "No GPU detected. On Colab, go to Runtime > Change runtime type > "
    "select a GPU (T4 / A100 / L4). Locally, install jax[cuda12]."
)

Imports

In [ ]:
import os
import math
import random
from functools import partial

import numpy as np
import jax
import jax.numpy as jnp
from jax import random as jrandom
import flax.linen as nn
import optax
from flax.training import train_state

from torchvision import datasets, transforms
import torch.utils.data

import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from scipy.special import softmax as scipy_softmax
import seaborn as sns
import imageio
from tqdm import tqdm

Set the seed for reproducibility

In [ ]:
def set_seed(seed=42):
    """Set random seeds for reproducibility across numpy and Python stdlib."""
    random.seed(seed)
    np.random.seed(seed)
    # JAX PRNG is handled explicitly via key passing, so we just create
    # the master key here and return it for downstream use.
    return jrandom.PRNGKey(seed)

master_key = set_seed(42)

We start by defining some helper classes, which we will use in the CTM.

The key building block is the `SuperLinear` layer, which implements N unique linear transformations -- one per neuron. In PyTorch this uses `nn.Parameter` and `torch.einsum`. In JAX/Flax, we define it as a `nn.Module` with explicit parameter initialization via `self.param(...)`.

**JAX-specific notes:**
- Flax modules use `@nn.compact` to define parameters inline during the first forward pass, similar to PyTorch's `LazyLinear` but with explicit shapes.
- Dropout in JAX requires a PRNG key passed at call time rather than being stateful, so we pass `deterministic=True/False` to control it.
- There is no `nn.Identity` needed -- we simply skip the dropout when `deterministic=True`.

In [ ]:
class SuperLinear(nn.Module):
    """SuperLinear Layer: Implements Neuron-Level Models (NLMs) for the CTM.

    Performs N independent linear transformations, one per neuron.
    Input shape:  (B, N, in_dims)  -- B=batch, N=neurons, in_dims=features per neuron
    Output shape: (B, N, out_dims)

    This is equivalent to the PyTorch version that uses:
        torch.einsum('BDM,MHD->BDH', x, w) + b
    where M=in_dims, H=out_dims, D=N (neurons).
    """
    out_dims: int
    N: int  # number of neurons
    dropout_rate: float = 0.0

    @nn.compact
    def __call__(self, x, deterministic: bool = True):
        in_dims = x.shape[-1]
        scale = 1.0 / math.sqrt(in_dims + self.out_dims)

        # Weight: (in_dims, out_dims, N) -- same layout as PyTorch version
        w = self.param(
            'w',
            lambda rng, shape: jrandom.uniform(rng, shape, minval=-scale, maxval=scale),
            (in_dims, self.out_dims, self.N),
        )
        # Bias: (1, N, out_dims)
        b = self.param('b', nn.initializers.zeros_init(), (1, self.N, self.out_dims))

        out = nn.Dropout(rate=self.dropout_rate)(x, deterministic=deterministic)
        # Einsum: (B, N, in_dims) x (in_dims, out_dims, N) -> (B, N, out_dims)
        out = jnp.einsum('BDM,MHD->BDH', out, w) + b
        # PyTorch squeeze(-1) is a no-op when dim != 1; JAX raises an error.
        if out.shape[-1] == 1:
            out = out.squeeze(-1)
        return out


class Squeeze(nn.Module):
    """Squeeze a specific dimension (unused -- inlined in the CTM forward pass)."""
    dim: int

    @nn.compact
    def __call__(self, x):
        return x.squeeze(self.dim)

Next, we define a helper function `compute_normalized_entropy`. We will use this function inside the CTM to compute the certainty of the model at each internal tick as `certainty = 1 - normalized entropy`.

In JAX this function is pure (no side effects) and will be JIT-compiled as part of the model's forward pass. We use `jax.nn.softmax` and `jax.nn.log_softmax` in place of their PyTorch counterparts.

In [ ]:
def compute_normalized_entropy(logits, reduction='mean'):
    """Computes the normalized entropy for certainty-loss.

    Args:
        logits: (B, num_classes) or (B, num_classes, T) shaped array of raw logits.
        reduction: If 'mean' and logits has more than 2 dims, flatten and average.

    Returns:
        Normalized entropy in [0, 1] per sample. Shape: (B,).
    """
    preds = jax.nn.softmax(logits, axis=-1)
    log_preds = jax.nn.log_softmax(logits, axis=-1)
    entropy = -jnp.sum(preds * log_preds, axis=-1)
    num_classes = logits.shape[-1]
    max_entropy = jnp.log(jnp.array(num_classes, dtype=jnp.float32))
    normalized_entropy = entropy / max_entropy
    if len(logits.shape) > 2 and reduction == 'mean':
        normalized_entropy = normalized_entropy.reshape(normalized_entropy.shape[0], -1).mean(-1)
    return normalized_entropy

## CTM Architecture Overview

The CTM implementation is initialized with the following core parameters:

- `iterations`: Number of internal ticks (recurrent steps).
- `d_model`: Total number of neurons.
- `d_input`: Input and attention embedding dimension.
- `memory_length`: Length of the sliding activation window used by each neuron.
- `heads`: Number of attention heads.
- `n_synch_out`: Number of neurons used for output synchronization.
- `n_synch_action`: Number of neurons used for computing attention queries.
- `out_dims`: Dimensionality of the model's output.

### Key Components

Upon initialization, the CTM builds the following modules:

- **Backbone**: A CNN feature extractor for the input (e.g. image).
- **Synapses**: A communication layer allowing neurons to interact.
- **Trace Processor**: A neuron-level model that operates on each neuron's temporal activation trace.
- **Synchronization Buffers**: For tracking decay.
- **Learned Initial States**: Starting activations and traces for the system.

---

## Forward Pass Mechanics

At each internal tick `stepi`, the CTM executes the following procedure:

1. **Initialize recurrent state**:
    - `state_trace`: Memory trace per neuron.
    - `activated_state`: Current post-activations.
    - `decay_alpha_out`, `decay_beta_out`: Values for calculating synchronization.

2. **Featurize input**:
    - Use the CNN backbone to extract key-value attention pairs `kv`.

3. **Internal Loop** (for each tick `stepi`):
    1. Compute `synchronisation_action` from `n_synch_action` neurons.
    2. Generate attention query `q` from this synchronization.
    3. Perform multi-head cross-attention over `kv`.
    4. Concatenate attention output with the current neuron activations.
    5. Update neurons via the synaptic module.
    6. Append new activation to the trace window.
    7. Update neuron states using the `trace_processor`.
    8. Compute `synchronisation_out` from `n_synch_out` neurons.
    9. Project to the output space via `output_projector`.
    10. Compute prediction certainty from normalized entropy.

This inner loop is repeated for the configured number of internal ticks. The CTM emits **predictions and certainties at every internal tick**.

### JAX-Specific Design Decisions

- The recurrent loop uses a **Python for-loop** that is **unrolled during JIT tracing**. This means XLA sees the full unrolled computation graph and can optimize across all iterations as a single fused kernel. For the small iteration counts used in practice (e.g. 15), this is efficient and keeps the code simple. A `jax.lax.scan`-based version would reduce compilation time and memory for larger iteration counts.
- Upper-triangular indices for synchronization are precomputed once and stored as module attributes, avoiding recomputation inside the loop.
- Multi-head attention is implemented manually with einsum operations, including learned Q/K/V projections and an output projection matching PyTorch's `nn.MultiheadAttention` semantics. This gives full JIT transparency with no opaque library calls.
- All parameter shapes are declared explicitly -- no lazy initialization.
- Decay rates use broadcasting rather than materialized copies to avoid unnecessary memory allocation.

> For detailed mathematical formulation of the synchronization mechanism, please refer to the technical report.

In [ ]:
class ContinuousThoughtMachine(nn.Module):
    """Continuous Thought Machine implemented in Flax/JAX.

    The recurrent loop uses a Python for-loop that is unrolled at JIT trace
    time, producing a single fused XLA computation for all iterations.
    Multi-head attention is implemented manually with einsum for full JIT
    transparency, including learned Q/K/V projections and an output projection
    matching PyTorch's nn.MultiheadAttention.
    """
    iterations: int
    d_model: int
    d_input: int
    memory_length: int
    heads: int
    n_synch_out: int
    n_synch_action: int
    out_dims: int
    memory_hidden_dims: int
    dropout_rate: float = 0.0

    def setup(self):
        d = self.d_input
        # --- Input Processing (CNN Backbone) ---
        self.conv1 = nn.Conv(features=d, kernel_size=(3, 3), strides=(1, 1), padding='SAME')
        self.bn1 = nn.BatchNorm()
        self.conv2 = nn.Conv(features=d, kernel_size=(3, 3), strides=(1, 1), padding='SAME')
        self.bn2 = nn.BatchNorm()

        # --- External projections ---
        self.kv_proj = nn.Dense(features=d)
        self.kv_ln = nn.LayerNorm()
        self.q_proj_linear = nn.Dense(features=d)

        # --- Internal attention projections (matching nn.MultiheadAttention) ---
        assert d % self.heads == 0, "d_input must be divisible by heads"
        self.head_dim = d // self.heads
        self.attn_q_proj = nn.Dense(features=d)
        self.attn_k_proj = nn.Dense(features=d)
        self.attn_v_proj = nn.Dense(features=d)
        self.attn_out_proj = nn.Dense(features=d)

        # --- Core CTM Modules ---
        self.synapse_drop = nn.Dropout(rate=self.dropout_rate)
        self.synapse_linear = nn.Dense(features=self.d_model * 2)
        self.synapse_ln = nn.LayerNorm()

        self.trace_sl1 = SuperLinear(
            out_dims=2 * self.memory_hidden_dims,
            N=self.d_model,
            dropout_rate=self.dropout_rate,
        )
        self.trace_sl2 = SuperLinear(
            out_dims=2,
            N=self.d_model,
            dropout_rate=self.dropout_rate,
        )

        # --- Output ---
        self.output_projector = nn.Dense(features=self.out_dims)

        # --- Synchronisation sizes ---
        self.synch_repr_size_action = (self.n_synch_action * (self.n_synch_action + 1)) // 2
        self.synch_repr_size_out = (self.n_synch_out * (self.n_synch_out + 1)) // 2

        # Precompute upper-triangular indices (used in compute_synchronisation)
        self.triu_idx_action = jnp.triu_indices(self.n_synch_action)
        self.triu_idx_out = jnp.triu_indices(self.n_synch_out)

        # --- Learned initial states ---
        self.start_activated_state = self.param(
            'start_activated_state',
            lambda rng, shape: jrandom.uniform(
                rng, shape,
                minval=-1.0 / math.sqrt(self.d_model),
                maxval=1.0 / math.sqrt(self.d_model),
            ),
            (self.d_model,),
        )
        self.start_trace = self.param(
            'start_trace',
            lambda rng, shape: jrandom.uniform(
                rng, shape,
                minval=-1.0 / math.sqrt(self.d_model + self.memory_length),
                maxval=1.0 / math.sqrt(self.d_model + self.memory_length),
            ),
            (self.d_model, self.memory_length),
        )

        # --- Decay parameters ---
        self.decay_params_action = self.param(
            'decay_params_action',
            nn.initializers.zeros_init(),
            (self.synch_repr_size_action,),
        )
        self.decay_params_out = self.param(
            'decay_params_out',
            nn.initializers.zeros_init(),
            (self.synch_repr_size_out,),
        )

    # -----------------------------------------------------------------
    # Helpers
    # -----------------------------------------------------------------

    def _max_pool_2x2(self, x):
        """2x2 max pooling using lax.reduce_window (JIT-friendly)."""
        return nn.max_pool(x, window_shape=(2, 2), strides=(2, 2), padding='VALID')

    def compute_features(self, x, use_running_average: bool):
        """CNN backbone: two conv blocks with BN, ReLU, and 2x2 max-pool."""
        h = self.conv1(x)
        h = self.bn1(h, use_running_average=use_running_average)
        h = nn.relu(h)
        h = self._max_pool_2x2(h)
        h = self.conv2(h)
        h = self.bn2(h, use_running_average=use_running_average)
        h = nn.relu(h)
        h = self._max_pool_2x2(h)
        # h: (B, H', W', d_input) -> flatten spatial -> (B, num_positions, d_input)
        B = h.shape[0]
        h = h.reshape(B, -1, self.d_input)
        kv = self.kv_ln(self.kv_proj(h))
        return kv

    def compute_synchronisation(self, activated_state, decay_alpha, decay_beta, r, synch_type):
        """Compute pairwise synchronization from selected neurons."""
        if synch_type == 'action':
            n = self.n_synch_action
            selected = activated_state[:, -n:]
            idx = self.triu_idx_action
        else:
            n = self.n_synch_out
            selected = activated_state[:, :n]
            idx = self.triu_idx_out

        # Outer product -> upper-triangular entries
        outer = selected[:, :, None] * selected[:, None, :]  # (B, n, n)
        pairwise_product = outer[:, idx[0], idx[1]]  # (B, synch_size)

        is_init = (decay_alpha is None)
        if is_init:
            decay_alpha = pairwise_product
            decay_beta = jnp.ones_like(pairwise_product)
        else:
            decay_alpha = r * decay_alpha + pairwise_product
            decay_beta = r * decay_beta + 1.0

        synchronisation = decay_alpha / jnp.sqrt(decay_beta)
        return synchronisation, decay_alpha, decay_beta

    def multi_head_attention(self, q, kv):
        """Manual multi-head cross-attention with learned projections.

        Matches PyTorch nn.MultiheadAttention semantics: applies separate
        learned linear projections for Q, K, V, then scaled dot-product
        attention, then an output projection.

        Args:
            q: (B, 1, d_input) query (already externally projected)
            kv: (B, S, d_input) keys/values (already externally projected)
        Returns:
            attn_out: (B, 1, d_input)
            attn_weights: (B, heads, 1, S)
        """
        B, S, _ = kv.shape
        H = self.heads
        D = self.head_dim

        # Internal learned projections (matching nn.MultiheadAttention)
        q = self.attn_q_proj(q)    # (B, 1, d_input)
        k = self.attn_k_proj(kv)   # (B, S, d_input)
        v = self.attn_v_proj(kv)   # (B, S, d_input)

        # Reshape to multi-head format
        q_heads = q.reshape(B, 1, H, D).transpose(0, 2, 1, 3)   # (B, H, 1, D)
        k_heads = k.reshape(B, S, H, D).transpose(0, 2, 1, 3)   # (B, H, S, D)
        v_heads = v.reshape(B, S, H, D).transpose(0, 2, 1, 3)   # (B, H, S, D)

        scale = jnp.sqrt(jnp.array(D, dtype=q.dtype))
        attn_logits = jnp.einsum('bhqd,bhkd->bhqk', q_heads, k_heads) / scale  # (B,H,1,S)
        attn_weights = jax.nn.softmax(attn_logits, axis=-1)

        attn_out = jnp.einsum('bhqk,bhkd->bhqd', attn_weights, v_heads)  # (B,H,1,D)
        attn_out = attn_out.transpose(0, 2, 1, 3).reshape(B, 1, self.d_input)  # (B,1,d_input)

        # Output projection (matching nn.MultiheadAttention.out_proj)
        attn_out = self.attn_out_proj(attn_out)  # (B, 1, d_input)

        return attn_out, attn_weights

    def compute_certainty(self, current_prediction):
        ne = compute_normalized_entropy(current_prediction)
        return jnp.stack([ne, 1.0 - ne], axis=-1)

    # -----------------------------------------------------------------
    # Forward pass
    # -----------------------------------------------------------------

    def __call__(self, x, deterministic: bool = True, track: bool = False):
        """Forward pass of the CTM.

        Args:
            x: Input images, shape (B, H, W, C).  Note: JAX uses channels-last.
            deterministic: If True, disable dropout (inference mode).
            track: If True, collect per-tick diagnostics for visualization.

        Returns:
            predictions: (B, out_dims, iterations)
            certainties: (B, 2, iterations)
            extra: synchronisation_out (default) or full tracking tuple
        """
        B = x.shape[0]
        use_running_average = deterministic

        # Clamp decay params (matching the PyTorch forward pre-hook)
        decay_params_action = jnp.clip(self.decay_params_action, 0.0, 15.0)
        decay_params_out = jnp.clip(self.decay_params_out, 0.0, 15.0)

        # --- Featurise input ---
        kv = self.compute_features(x, use_running_average=use_running_average)

        # --- Initialise recurrent state ---
        state_trace = jnp.broadcast_to(
            self.start_trace[None, :, :], (B, self.d_model, self.memory_length)
        )
        activated_state = jnp.broadcast_to(
            self.start_activated_state[None, :], (B, self.d_model)
        )

        # Decay rates -- broadcast-compatible with (B, synch_size), no .repeat() needed
        r_action = jnp.exp(-decay_params_action)[None, :]  # (1, synch_size_action)
        r_out = jnp.exp(-decay_params_out)[None, :]        # (1, synch_size_out)

        # Initial synchronisation_out (to seed decay_alpha/beta)
        _, decay_alpha_out, decay_beta_out = self.compute_synchronisation(
            activated_state, None, None, r_out, synch_type='out'
        )
        decay_alpha_action = None
        decay_beta_action = None

        # --- Recurrent loop ---
        # Uses a Python for-loop that is unrolled during JIT tracing, producing
        # a single fused XLA computation. This is efficient for the small number
        # of iterations used in practice (e.g. 15).
        all_predictions = []
        all_certainties = []

        # Tracking storage
        pre_acts_track = []
        post_acts_track = []
        attn_track = []
        synch_out_track = []
        synch_action_track = []

        for stepi in range(self.iterations):
            # 1. Synchronisation for attention query
            synch_action, decay_alpha_action, decay_beta_action = self.compute_synchronisation(
                activated_state, decay_alpha_action, decay_beta_action, r_action, synch_type='action'
            )

            # 2. Attention
            q = self.q_proj_linear(synch_action)[:, None, :]  # (B, 1, d_input)
            attn_out, attn_weights = self.multi_head_attention(q, kv)
            attn_out = attn_out.squeeze(1)  # (B, d_input)

            # 3. Synapses
            pre_synapse = jnp.concatenate([attn_out, activated_state], axis=-1)
            state = self.synapse_drop(pre_synapse, deterministic=deterministic)
            state = self.synapse_linear(state)
            # GLU activation: split in half
            a, b = jnp.split(state, 2, axis=-1)
            state = a * jax.nn.sigmoid(b)
            state = self.synapse_ln(state)

            # 4. Update trace
            state_trace = jnp.concatenate(
                [state_trace[:, :, 1:], state[:, :, None]], axis=-1
            )

            # 5. Activate via trace processor (two SuperLinear + GLU layers)
            tp = self.trace_sl1(state_trace, deterministic=deterministic)
            tp_a, tp_b = jnp.split(tp, 2, axis=-1)
            tp = tp_a * jax.nn.sigmoid(tp_b)
            tp = self.trace_sl2(tp, deterministic=deterministic)
            tp_a2, tp_b2 = jnp.split(tp, 2, axis=-1)
            activated_state = (tp_a2 * jax.nn.sigmoid(tp_b2)).squeeze(-1)

            # 6. Synchronisation for output
            synch_out, decay_alpha_out, decay_beta_out = self.compute_synchronisation(
                activated_state, decay_alpha_out, decay_beta_out, r_out, synch_type='out'
            )

            # 7. Prediction + certainty
            current_prediction = self.output_projector(synch_out)
            current_certainty = self.compute_certainty(current_prediction)

            all_predictions.append(current_prediction)
            all_certainties.append(current_certainty)

            if track:
                pre_acts_track.append(state_trace[:, :, -1])
                post_acts_track.append(activated_state)
                attn_track.append(attn_weights)
                synch_out_track.append(synch_out)
                synch_action_track.append(synch_action)

        # Stack along the iterations axis: (B, out_dims, T) and (B, 2, T)
        predictions = jnp.stack(all_predictions, axis=-1)
        certainties = jnp.stack(all_certainties, axis=-1)

        if track:
            return (
                predictions,
                certainties,
                (jnp.stack(synch_out_track), jnp.stack(synch_action_track)),
                jnp.stack(pre_acts_track),
                jnp.stack(post_acts_track),
                jnp.stack(attn_track),
            )
        return predictions, certainties, synch_out

## Certainty-Based Loss Function

The CTM produces outputs at each internal tick, so the question arises: **how do we optimize the model across this internal temporal dimension?**

Our answer is a simple but effective **certainty-based loss** that encourages the model to reason meaningfully across time. Instead of relying on the final tick alone, we aggregate loss from two key internal ticks:

1. The tick where the **prediction loss** is lowest.
2. The tick where the **certainty** (1 - normalized entropy) is highest.

We then take the **average of the losses** at these two points.

This approach encourages the CTM to both make accurate predictions and express high confidence in them -- while supporting adaptive, interpretable computation over time.

### JAX-Specific Notes

- Cross-entropy is computed with `optax.softmax_cross_entropy_with_integer_labels` which takes integer targets directly (no one-hot encoding needed).
- We use `jnp.argmin` / `jnp.argmax` instead of PyTorch's `.argmin()` / `.argmax()`.
- Advanced indexing with `jnp.arange` + gathered indices replaces PyTorch's `batch_indexer` pattern.

In [ ]:
def get_loss(predictions, certainties, targets, use_most_certain=True):
    """Certainty-based loss: average of loss at best-prediction tick and most-certain tick.

    Args:
        predictions: (B, num_classes, T) logits at each internal tick.
        certainties: (B, 2, T) where index 1 is the certainty (1 - norm_entropy).
        targets: (B,) integer class labels.
        use_most_certain: If False, use the final tick instead of the most certain one.

    Returns:
        loss: scalar
        loss_index_2: (B,) indices of the selected tick (most certain or final)
    """
    B, C, T = predictions.shape

    # Compute per-tick cross-entropy: (B, T)
    # predictions is (B, C, T), we need (B, T, C) for the CE function
    preds_btc = jnp.transpose(predictions, (0, 2, 1))  # (B, T, C)
    targets_rep = jnp.broadcast_to(targets[:, None], (B, T))  # (B, T)
    losses = optax.softmax_cross_entropy_with_integer_labels(preds_btc, targets_rep)  # (B, T)

    # Tick with lowest loss
    loss_index_1 = jnp.argmin(losses, axis=1)  # (B,)

    # Tick with highest certainty (or final tick)
    if use_most_certain:
        loss_index_2 = jnp.argmax(certainties[:, 1, :], axis=-1)  # (B,)
    else:
        loss_index_2 = jnp.full((B,), T - 1, dtype=jnp.int32)

    batch_idx = jnp.arange(B)
    loss_minimum_ce = jnp.mean(losses[batch_idx, loss_index_1])
    loss_selected = jnp.mean(losses[batch_idx, loss_index_2])

    loss = (loss_minimum_ce + loss_selected) / 2.0
    return loss, loss_index_2

In [ ]:
def calculate_accuracy(predictions, targets, where_most_certain):
    """Calculate accuracy based on the prediction at the most certain internal tick.

    Args:
        predictions: (B, num_classes, T) logits.
        targets: (B,) integer class labels.
        where_most_certain: (B,) tick indices.

    Returns:
        accuracy: scalar float.
    """
    B = predictions.shape[0]
    # predicted class at each tick: (B, T)
    pred_classes = jnp.argmax(predictions, axis=1)
    # gather the prediction at the most-certain tick for each sample
    selected_preds = pred_classes[jnp.arange(B), where_most_certain]
    accuracy = jnp.mean(selected_preds == targets)
    return accuracy

In [ ]:
def prepare_data(batch_size=256):
    """Prepare MNIST train and test data loaders using torchvision.

    We use torchvision for data loading and augmentation, then convert
    PyTorch tensors to JAX arrays in the training loop. Images are returned
    in channels-last format (H, W, C) as expected by JAX convolutions.
    """
    transform = transforms.Compose([
        transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])

    # Test transform without augmentation for consistent evaluation
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])

    train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_data = datasets.MNIST(root="./data", train=False, download=True, transform=test_transform)

    trainloader = torch.utils.data.DataLoader(
        train_data, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True
    )
    testloader = torch.utils.data.DataLoader(
        test_data, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False
    )
    return trainloader, testloader


def torch_to_jax(inputs, targets):
    """Convert a PyTorch batch to JAX arrays with channels-last layout.

    PyTorch images are (B, C, H, W) -> transpose to (B, H, W, C) for JAX.
    """
    images = jnp.array(inputs.numpy()).transpose(0, 2, 3, 1)  # (B, H, W, C)
    labels = jnp.array(targets.numpy(), dtype=jnp.int32)
    return images, labels

In [ ]:
def update_training_curve_plot(fig, ax1, ax2, train_losses, test_losses,
                               train_accuracies, test_accuracies, steps):
    """Update the live training curve plot (same layout as the PyTorch version)."""
    clear_output(wait=True)

    ax1.clear()
    ax1.plot(range(len(train_losses)), train_losses, 'b-', alpha=0.7,
             label=f'Train Loss: {train_losses[-1]:.3f}')
    ax1.plot(steps, test_losses, 'r-', marker='o',
             label=f'Test Loss: {test_losses[-1]:.3f}')
    ax1.set_title('Loss')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.clear()
    ax2.plot(range(len(train_accuracies)), train_accuracies, 'b-', alpha=0.7,
             label=f'Train Accuracy: {train_accuracies[-1]:.3f}')
    ax2.plot(steps, test_accuracies, 'r-', marker='o',
             label=f'Test Accuracy: {test_accuracies[-1]:.3f}')
    ax2.set_title('Accuracy')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    display(fig)

In [ ]:
# -------------------------------------------------------------------------
# Flax TrainState subclass that also carries batch_stats (for BatchNorm)
# -------------------------------------------------------------------------
class TrainState(train_state.TrainState):
    """Extended TrainState that carries mutable batch_stats for BatchNorm."""
    batch_stats: any


# -------------------------------------------------------------------------
# JIT-compiled training step
# -------------------------------------------------------------------------
@jax.jit
def train_step(state, images, labels):
    """Single training step: forward, loss, backward, update.

    This entire function is JIT-compiled by XLA, including the CTM's
    internal recurrent loop (which is unrolled at trace time).

    Args:
        state: TrainState with params, batch_stats, opt_state, etc.
        images: (B, H, W, C) input images.
        labels: (B,) integer targets.

    Returns:
        Updated state, loss scalar, accuracy scalar, most-certain indices.
    """
    # Derive a per-step dropout key so dropout is deterministic but varies
    # across training steps. Uses fold_in(base_key, step) for zero overhead.
    dropout_rng = jrandom.fold_in(jrandom.PRNGKey(0), state.step)

    def loss_fn(params):
        (predictions, certainties, _), updates = state.apply_fn(
            {'params': params, 'batch_stats': state.batch_stats},
            images,
            deterministic=False,
            mutable=['batch_stats'],
            rngs={'dropout': dropout_rng},
        )
        loss, where_most_certain = get_loss(predictions, certainties, labels)
        return loss, (predictions, certainties, where_most_certain, updates)

    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    (loss, (predictions, certainties, where_most_certain, updates)), grads = grad_fn(state.params)

    # Apply gradients first, then clamp decay params (matching PyTorch pre-hook)
    state = state.apply_gradients(grads=grads)
    state = state.replace(
        batch_stats=updates['batch_stats'],
        params={
            **state.params,
            'decay_params_action': jnp.clip(state.params['decay_params_action'], 0.0, 15.0),
            'decay_params_out': jnp.clip(state.params['decay_params_out'], 0.0, 15.0),
        },
    )

    accuracy = calculate_accuracy(predictions, labels, where_most_certain)
    return state, loss, accuracy, where_most_certain


# -------------------------------------------------------------------------
# JIT-compiled evaluation step
# -------------------------------------------------------------------------
@jax.jit
def eval_step(state, images, labels):
    """Single evaluation step (no gradients, BN in inference mode).

    Args:
        state: TrainState.
        images: (B, H, W, C).
        labels: (B,).

    Returns:
        loss, predictions, where_most_certain.
    """
    predictions, certainties, _ = state.apply_fn(
        {'params': state.params, 'batch_stats': state.batch_stats},
        images,
        deterministic=True,
    )
    loss, where_most_certain = get_loss(predictions, certainties, labels)
    return loss, predictions, certainties, where_most_certain

In [ ]:
def train(state, trainloader, testloader, iterations, test_every):
    """Main training loop with periodic evaluation and live plotting.

    Args:
        state: Flax TrainState (params + optimizer + batch_stats).
        trainloader: PyTorch DataLoader for training data.
        testloader: PyTorch DataLoader for test data.
        iterations: Total number of training steps.
        test_every: Evaluate on the full test set every this many steps.

    Returns:
        Updated TrainState after training.
    """
    iterator = iter(trainloader)

    train_losses = []
    test_losses = []
    train_accuracies = []
    test_accuracies = []
    steps = []

    plt.ion()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    test_loss = None
    test_accuracy = None

    with tqdm(total=iterations, initial=0, dynamic_ncols=True) as pbar:
        for stepi in range(iterations):
            # --- Get next batch (cycle through DataLoader) ---
            try:
                inputs, targets = next(iterator)
            except StopIteration:
                iterator = iter(trainloader)
                inputs, targets = next(iterator)

            images, labels = torch_to_jax(inputs, targets)

            # --- Train step ---
            state, t_loss, t_acc, _ = train_step(state, images, labels)
            train_losses.append(float(t_loss))
            train_accuracies.append(float(t_acc))

            # --- Periodic evaluation ---
            if stepi % test_every == 0 or stepi == iterations - 1:
                all_test_losses = []
                all_test_preds = []
                all_test_targets = []
                all_test_where = []

                for test_inputs, test_targets in testloader:
                    test_images, test_labels = torch_to_jax(test_inputs, test_targets)
                    e_loss, e_preds, e_certs, e_where = eval_step(
                        state, test_images, test_labels
                    )
                    all_test_losses.append(float(e_loss))
                    all_test_preds.append(e_preds)
                    all_test_targets.append(test_labels)
                    all_test_where.append(e_where)

                all_test_preds = jnp.concatenate(all_test_preds, axis=0)
                all_test_targets = jnp.concatenate(all_test_targets, axis=0)
                all_test_where = jnp.concatenate(all_test_where, axis=0)

                test_accuracy = float(
                    calculate_accuracy(all_test_preds, all_test_targets, all_test_where)
                )
                test_loss = sum(all_test_losses) / len(all_test_losses)

                test_losses.append(test_loss)
                test_accuracies.append(test_accuracy)
                steps.append(stepi)

                update_training_curve_plot(
                    fig, ax1, ax2,
                    train_losses, test_losses,
                    train_accuracies, test_accuracies,
                    steps,
                )

            pbar.set_description(
                f'Train Loss: {t_loss:.3f}, Train Acc: {t_acc:.3f} '
                f'Test Loss: {test_loss}, Test Acc: {test_accuracy}'
            )
            pbar.update(1)

    plt.ioff()
    plt.close(fig)
    return state

In [ ]:
# --- Prepare data ---
trainloader, testloader = prepare_data(batch_size=256)

# --- Instantiate model ---
model = ContinuousThoughtMachine(
    iterations=15,
    d_model=128,
    d_input=128,
    memory_length=10,
    heads=2,
    n_synch_out=16,
    n_synch_action=16,
    memory_hidden_dims=8,
    out_dims=10,
    dropout_rate=0.0,
)

# --- Initialize parameters with a dummy input ---
# JAX uses channels-last: (B, H, W, C) for a 28x28 grayscale image
dummy_input = jnp.ones((1, 28, 28, 1))
key, init_key = jrandom.split(master_key)
variables = model.init(init_key, dummy_input, deterministic=True)

params = variables['params']
batch_stats = variables['batch_stats']

# --- Create optimizer and TrainState ---
optimizer = optax.adamw(learning_rate=1e-4, eps=1e-8)

state = TrainState.create(
    apply_fn=model.apply,
    params=params,
    tx=optimizer,
    batch_stats=batch_stats,
)

# --- Print parameter count ---
param_count = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f'Model parameters: {param_count:,}')

In [ ]:
state = train(
    state=state,
    trainloader=trainloader,
    testloader=testloader,
    iterations=10000,
    test_every=500,
)

## Visualizing CTM Dynamics

We define a function to create GIFs that show how the CTM's dynamics evolve over internal ticks. These visualizations include:

- **Neuron activations** at each internal tick
- **Attention patterns** across the input
- **Predictions and certainty** at every step

The top row of the gif shows the input image, the attention weights and the model's predictions.

The second plot, in black, shows the model's certainty over time.

The red and blue lines correspond to the post-activations of different neurons, with the corresponding pre-activation shown in gray.

In [ ]:
def make_gif(predictions, certainties, targets, pre_activations, post_activations,
             attention, inputs_to_model, filename):
    """Create a GIF visualizing CTM dynamics across internal ticks.

    All inputs should be numpy arrays (moved off device before calling).

    Args:
        predictions: (B, num_classes, T)
        certainties: (B, 2, T)
        targets: (B,)
        pre_activations: (T, B, d_model)
        post_activations: (T, B, d_model)
        attention: (T, B, heads, 1, num_positions)
        inputs_to_model: (B, H, W, C)  -- channels-last
        filename: output gif path
    """
    def reshape_attention_weights(attention, target_size=28):
        # attention: (T, B, num_heads, 1, num_positions)
        T, B, num_heads, _, num_positions = attention.shape
        # Average over heads and squeeze the query dim
        attention = attention.mean(axis=2).squeeze(2)  # (T, B, num_positions)
        height = int(num_positions ** 0.5)
        while num_positions % height != 0:
            height -= 1
        width = num_positions // height
        attention = attention.reshape(T, B, height, width)
        # Resize to target_size x target_size using scipy/numpy (no torch dependency)
        from scipy.ndimage import zoom
        result = np.zeros((T, B, target_size, target_size))
        for t in range(T):
            for b_idx in range(B):
                result[t, b_idx] = zoom(attention[t, b_idx],
                                        (target_size / height, target_size / width),
                                        order=1)
        return result

    batch_index = 0
    n_neurons_to_visualise = 16
    figscale = 0.28
    n_steps = len(pre_activations)
    heatmap_cmap = sns.color_palette("viridis", as_cmap=True)
    frames = []

    attention = reshape_attention_weights(attention)

    these_pre_acts = pre_activations[:, batch_index, :]
    these_post_acts = post_activations[:, batch_index, :]
    # inputs_to_model is (B, H, W, C) channels-last
    these_inputs = inputs_to_model[batch_index, :, :, :]
    these_attention_weights = attention[:, batch_index, :, :]
    these_predictions = predictions[batch_index, :, :]
    these_certainties = certainties[batch_index, :, :]
    this_target = targets[batch_index]

    class_labels = [str(i) for i in range(these_predictions.shape[0])]

    mosaic = ([['img_data', 'img_data', 'attention', 'attention',
                'probs', 'probs', 'probs', 'probs']] * 4 +
              [['certainty'] * 8] +
              [[f'trace_{ti}'] * 8 for ti in range(n_neurons_to_visualise)])

    for stepi in tqdm(range(n_steps), desc="Processing steps", unit="step"):
        fig_gif, axes_gif = plt.subplot_mosaic(
            mosaic=mosaic,
            figsize=(31 * figscale * 8 / 4, 76 * figscale),
        )
        probs = scipy_softmax(these_predictions[:, stepi])
        colors = [('g' if i == this_target else 'b') for i in range(len(probs))]

        axes_gif['probs'].bar(np.arange(len(probs)), probs, color=colors, width=0.9, alpha=0.5)
        axes_gif['probs'].set_title('Probabilities')
        axes_gif['probs'].set_xticks(np.arange(len(probs)))
        axes_gif['probs'].set_xticklabels(class_labels, fontsize=24)
        axes_gif['probs'].set_yticks([])
        axes_gif['probs'].tick_params(left=False, bottom=False)
        axes_gif['probs'].set_ylim([0, 1])
        for spine in axes_gif['probs'].spines.values():
            spine.set_visible(False)

        # Certainty
        axes_gif['certainty'].plot(np.arange(n_steps), these_certainties[1], 'k-', linewidth=2)
        axes_gif['certainty'].set_xlim([0, n_steps - 1])
        axes_gif['certainty'].axvline(x=stepi, color='black', linewidth=1, alpha=0.5)
        axes_gif['certainty'].set_xticklabels([])
        axes_gif['certainty'].set_yticklabels([])
        axes_gif['certainty'].grid(False)
        for spine in axes_gif['certainty'].spines.values():
            spine.set_visible(False)

        # Neuron Traces
        for neuroni in range(n_neurons_to_visualise):
            ax = axes_gif[f'trace_{neuroni}']
            pre_activation = these_pre_acts[:, neuroni]
            post_activation = these_post_acts[:, neuroni]
            ax_pre = ax.twinx()

            ax_pre.plot(np.arange(n_steps), pre_activation, color='grey',
                        linestyle='--', linewidth=1, alpha=0.4)
            color = 'blue' if neuroni % 2 else 'red'
            ax.plot(np.arange(n_steps), post_activation, color=color, linewidth=2, alpha=1.0)

            ax.set_xlim([0, n_steps - 1])
            ax_pre.set_xlim([0, n_steps - 1])
            ax.set_ylim([np.min(post_activation), np.max(post_activation)])
            ax_pre.set_ylim([np.min(pre_activation), np.max(pre_activation)])

            ax.axvline(x=stepi, color='black', linewidth=1, alpha=0.5)
            ax.set_xticklabels([])
            ax.set_yticklabels([])
            ax.grid(False)
            ax_pre.set_xticklabels([])
            ax_pre.set_yticklabels([])
            ax_pre.grid(False)

            for spine in ax.spines.values():
                spine.set_visible(False)
            for spine in ax_pre.spines.values():
                spine.set_visible(False)

        # Input image -- channels-last, single channel
        this_image = these_inputs[:, :, 0]
        this_image = (this_image - this_image.min()) / (this_image.max() - this_image.min() + 1e-8)
        axes_gif['img_data'].set_title('Input Image')
        axes_gif['img_data'].imshow(this_image, cmap='binary', vmin=0, vmax=1)
        axes_gif['img_data'].axis('off')

        # Attention
        this_input_gate = these_attention_weights[stepi]
        gate_min, gate_max = np.nanmin(this_input_gate), np.nanmax(this_input_gate)
        if not np.isclose(gate_min, gate_max):
            normalized_gate = (this_input_gate - gate_min) / (gate_max - gate_min + 1e-8)
        else:
            normalized_gate = np.zeros_like(this_input_gate)
        attention_weights_heatmap = heatmap_cmap(normalized_gate)[:, :, :3]

        axes_gif['attention'].imshow(attention_weights_heatmap, vmin=0, vmax=1)
        axes_gif['attention'].axis('off')
        axes_gif['attention'].set_title('Attention')

        fig_gif.tight_layout()
        canvas = fig_gif.canvas
        canvas.draw()
        image_numpy = np.frombuffer(canvas.buffer_rgba(), dtype='uint8')
        image_numpy = image_numpy.reshape(*reversed(canvas.get_width_height()), 4)[:, :, :3]
        frames.append(image_numpy)
        plt.close(fig_gif)

    imageio.mimsave(filename, frames, fps=5, loop=100)
    print(f"GIF saved to {filename}")

In [ ]:
logdir = "mnist_logs"
os.makedirs(logdir, exist_ok=True)

# Run a forward pass with tracking enabled (not JIT -- uses Python loop)
inputs, targets = next(iter(testloader))
images, labels = torch_to_jax(inputs, targets)

(predictions, certainties,
 (synch_out_tracking, synch_action_tracking),
 pre_activations_tracking,
 post_activations_tracking,
 attention) = model.apply(
    {'params': state.params, 'batch_stats': state.batch_stats},
    images,
    deterministic=True,
    track=True,
)

# Convert to numpy for visualization
make_gif(
    np.array(predictions),
    np.array(certainties),
    np.array(labels),
    np.array(pre_activations_tracking),
    np.array(post_activations_tracking),
    np.array(attention),
    np.array(images),
    f"{logdir}/prediction.gif",
)